# Support Vector Machines: Maximum Margin Classification

Support Vector Machines (SVMs) are a set of supervised learning methods used for classification, regression, and outlier detection. The core idea is to find a hyperplane that best separates classes by maximizing the margin between them. SVMs are particularly effective in high-dimensional spaces and are versatile through the use of different kernel functions.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import ListedColormap
from sklearn.svm import SVC, SVR, LinearSVC
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.datasets import make_classification, make_blobs, make_circles, make_moons
from sklearn.datasets import load_digits, load_wine, load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import accuracy_score, f1_score, mean_squared_error
from sklearn.pipeline import Pipeline

sns.set_theme(style='whitegrid')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
np.random.seed(42)

---
## 1. Intuition: Finding the "Best" Separating Line

Among infinitely many lines that can separate two classes, SVM selects the one that **maximizes the margin** — the distance from the line to the closest points of each class. This maximum-margin hyperplane tends to generalize better.

In [ ]:
X, y = make_blobs(n_samples=50, centers=2, n_features=2, cluster_std=1.2, random_state=6)
y = 2 * y - 1  # convert to -1, 1

model = SVC(kernel='linear', C=1e5)
model.fit(X, y)

w = model.coef_[0]
b = model.intercept_[0]
xx = np.linspace(X[:, 0].min() - 1, X[:, 0].max() + 1, 200)
yy = -(w[0] * xx + b) / w[1]

# Distance from line = 1/||w|| -> offset
margin = 1 / np.sqrt(np.sum(w ** 2))
yy_upper = yy + margin / np.sqrt(w[0]**2 + w[1]**2)
yy_lower = yy - margin / np.sqrt(w[0]**2 + w[1]**2)

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(X[:, 0], X[:, 1], c=y, cmap='bwr', edgecolors='k', s=60)
ax.plot(xx, yy, 'k-', linewidth=2, label='Decision Boundary')
ax.plot(xx, yy_upper, 'k--', linewidth=1.5, label='Margin')
ax.plot(xx, yy_lower, 'k--', linewidth=1.5)
ax.scatter(model.support_vectors_[:, 0], model.support_vectors_[:, 1],
           s=180, facecolors='none', edgecolors='k', linewidth=2,
           label='Support Vectors')
ax.set_xlim(X[:, 0].min() - 1, X[:, 0].max() + 1)
ax.set_ylim(X[:, 1].min() - 1, X[:, 1].max() + 1)
ax.set_title('Maximum Margin Hyperplane with Support Vectors')
ax.set_xlabel('Feature 1')
ax.set_ylabel('Feature 2')
ax.legend()
plt.tight_layout()
plt.show()

---
## 2. What is the Margin? Support Vectors Explained

The **margin** is the perpendicular distance between the decision boundary and the closest data points from either class. These closest points are called **support vectors** — they alone define the hyperplane. If you move other points (non-support vectors), the decision boundary remains unchanged.

Mathematically, for a hyperplane $w \cdot x + b = 0$, the margin width is $2 / \|w\|$. SVM solves:

$$\min_{w,b} \frac{1}{2}\|w\|^2 \quad \text{subject to} \quad y_i(w \cdot x_i + b) \geq 1 \; \forall i$$

---
## 3. Hard Margin SVM (Perfectly Separable Data)

When the data is perfectly linearly separable, we can enforce that **no point lies inside the margin**. This is the hard-margin formulation. In practice, hard-margin SVMs are rarely used because real-world data is almost never perfectly separable, and the model becomes extremely sensitive to outliers.

In [ ]:
# Demonstrate hard margin on perfectly separable data
X_hard, y_hard = make_blobs(n_samples=30, centers=2, n_features=2, cluster_std=0.5, random_state=42)

svm_hard = SVC(kernel='linear', C=1e10)  # very large C -> hard margin
svm_hard.fit(X_hard, y_hard)

xx = np.linspace(X_hard[:, 0].min() - 1, X_hard[:, 0].max() + 1, 200)
w = svm_hard.coef_[0]
b = svm_hard.intercept_[0]
yy = -(w[0] * xx + b) / w[1]

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(X_hard[:, 0], X_hard[:, 1], c=y_hard, cmap='coolwarm', edgecolors='k', s=60)
ax.plot(xx, yy, 'k-', linewidth=2)
ax.scatter(svm_hard.support_vectors_[:, 0], svm_hard.support_vectors_[:, 1],
           s=180, facecolors='none', edgecolors='k', linewidth=2, label='Support Vectors')
ax.set_title('Hard Margin SVM (Perfectly Separable)')
ax.legend()
plt.tight_layout()
plt.show()

---
## 4. Soft Margin SVM (C Parameter)

Soft-margin SVM introduces **slack variables** $\xi_i$ to allow misclassifications. The parameter $C$ controls the trade-off:
- **Large C**: low tolerance for misclassification → narrower margin, may overfit
- **Small C**: higher tolerance → wider margin, may underfit

$$\min_{w,b,\xi} \frac{1}{2}\|w\|^2 + C \sum_i \xi_i$$

In [ ]:
X_soft, y_soft = make_blobs(n_samples=50, centers=2, n_features=2, cluster_std=2.5, random_state=3)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, C in zip(axes, [0.01, 1, 1000]):
    svm = SVC(kernel='linear', C=C)
    svm.fit(X_soft, y_soft)
    
    xx = np.linspace(X_soft[:, 0].min() - 1, X_soft[:, 0].max() + 1, 200)
    w = svm.coef_[0]
    b = svm.intercept_[0]
    yy = -(w[0] * xx + b) / w[1]
    
    ax.scatter(X_soft[:, 0], X_soft[:, 1], c=y_soft, cmap='coolwarm', edgecolors='k', s=50)
    ax.plot(xx, yy, 'k-', linewidth=2)
    ax.scatter(svm.support_vectors_[:, 0], svm.support_vectors_[:, 1],
               s=150, facecolors='none', edgecolors='k', linewidth=1.5, label='SVs')
    ax.set_title(f'C = {C}')
    ax.set_xlim(X_soft[:, 0].min() - 1, X_soft[:, 0].max() + 1)
    ax.set_ylim(X_soft[:, 1].min() - 1, X_soft[:, 1].max() + 1)
    ax.legend(fontsize=9)

plt.suptitle('Effect of C on Margin Width', fontsize=14)
plt.tight_layout()
plt.show()

---
## 5. The Kernel Trick: Mapping to Higher Dimensions

When data is not linearly separable in the original space, the **kernel trick** implicitly maps it to a higher-dimensional space where a linear separation becomes possible — without ever computing the coordinates in that space.

The kernel function $K(x_i, x_j) = \langle \phi(x_i), \phi(x_j) \rangle$ computes the dot product in the transformed space efficiently.

### Common Kernels

| Kernel | Formula | Notes |
|--------|---------|-------|
| Linear | $K(x_i, x_j) = x_i \cdot x_j$ | No transformation; equivalent to linear SVM |
| Polynomial | $K(x_i, x_j) = (\gamma x_i \cdot x_j + r)^d$ | Creates polynomial decision boundaries |
| RBF (Gaussian) | $K(x_i, x_j) = \exp(-\gamma \|x_i - x_j\|^2)$ | Most popular; infinite-dimensional mapping |
| Sigmoid | $K(x_i, x_j) = \tanh(\gamma x_i \cdot x_j + r)$ | Like a neural network activation |

In [ ]:
# Helper to plot decision boundary
def plot_decision_boundary(clf, X, y, ax, title=''):
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300),
                         np.linspace(y_min, y_max, 300))
    Z = clf.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='coolwarm')
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap='coolwarm', edgecolors='k', s=40)
    ax.set_xlim(xx.min(), xx.max())
    ax.set_ylim(yy.min(), yy.max())
    ax.set_title(title)
    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')

---
## 6. Kernel Comparison on Non-Linear Data

In [ ]:
X_circ, y_circ = make_circles(n_samples=200, noise=0.08, factor=0.4, random_state=42)
X_moon, y_moon = make_moons(n_samples=200, noise=0.12, random_state=42)

kernels = ['linear', 'poly', 'rbf', 'sigmoid']
kernel_labels = ['Linear', 'Polynomial (deg=3)', 'RBF (Gaussian)', 'Sigmoid']

fig, axes = plt.subplots(2, 4, figsize=(16, 7))

for row, (X_data, y_data, dset_name) in enumerate([
    (X_circ, y_circ, 'Concentric Circles'),
    (X_moon, y_moon, 'Moons')
]):
    for col, (kernel, label) in enumerate(zip(kernels, kernel_labels)):
        ax = axes[row, col]
        if kernel == 'poly':
            svm = SVC(kernel=kernel, degree=3, coef0=1, gamma='scale')
        elif kernel == 'sigmoid':
            svm = SVC(kernel=kernel, gamma='scale', coef0=1)
        else:
            svm = SVC(kernel=kernel, gamma='scale')
        svm.fit(X_data, y_data)
        plot_decision_boundary(svm, X_data, y_data, ax, f'{label}\n({dset_name})')

plt.suptitle('Kernel Comparison on Non-Linear Datasets', fontsize=15)
plt.tight_layout()
plt.show()

---
## 7. Gamma Parameter for RBF Kernel

The $\gamma$ (gamma) parameter defines how far the influence of a single training example reaches:
- **Low gamma**: large influence radius → smoother, more global decision boundary (high bias)
- **High gamma**: small influence radius → each point has a small sphere of influence, boundary follows data closely (high variance)

$$K(x_i, x_j) = \exp\left(-\gamma \|x_i - x_j\|^2\right)$$

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
gamma_values = [0.01, 0.1, 0.5, 1, 5, 50]

X_m, y_m = make_moons(n_samples=200, noise=0.15, random_state=42)

for ax, gamma in zip(axes.ravel(), gamma_values):
    svm = SVC(kernel='rbf', gamma=gamma, C=1)
    svm.fit(X_m, y_m)
    plot_decision_boundary(svm, X_m, y_m, ax, f'$\\gamma$ = {gamma}')
    train_acc = svm.score(X_m, y_m)
    ax.text(0.05, 0.95, f'Acc: {train_acc:.2f}', transform=ax.transAxes,
            va='top', fontsize=10, bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

plt.suptitle('Effect of Gamma on RBF Kernel SVM Decision Boundary', fontsize=14)
plt.tight_layout()
plt.show()

---
## 8. Support Vector Regression (SVR)

SVR extends SVM to regression. Instead of maximizing margin, it finds a function that deviates from the actual targets by at most $\varepsilon$ (epsilon). Points inside the **epsilon-insensitive tube** do not contribute to the loss.

$$\min \frac{1}{2}\|w\|^2 + C \sum_i (\xi_i + \xi_i^*) \quad \text{s.t.} \quad |y_i - (w \cdot x_i + b)| \leq \varepsilon + \xi_i$$

In [ ]:
X_svr = np.sort(5 * np.random.rand(80, 1), axis=0)
y_svr = np.sin(X_svr).ravel() + 0.2 * (0.5 - np.random.rand(80))

svr_rbf = SVR(kernel='rbf', C=10, gamma=1, epsilon=0.1)
svr_lin = SVR(kernel='linear', C=10, epsilon=0.1)
svr_poly = SVR(kernel='poly', degree=3, C=10, gamma='scale', epsilon=0.1)

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(X_svr, y_svr, color='navy', s=30, label='Data', alpha=0.7)

X_test = np.arange(0, 5, 0.01)[:, np.newaxis]
for name, model in [('RBF', svr_rbf), ('Linear', svr_lin), ('Polynomial (deg=3)', svr_poly)]:
    model.fit(X_svr, y_svr)
    y_pred = model.predict(X_test)
    ax.plot(X_test, y_pred, label=f'{name} SVR (MSE={mean_squared_error(np.sin(X_test).ravel(), y_pred):.3f})', lw=2)

ax.set_xlabel('X')
ax.set_ylabel('y')
ax.set_title('Support Vector Regression with Different Kernels')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

---
## 9. Feature Scaling is Crucial for SVM

SVM is **scale-sensitive** because it relies on distances between points. Features with larger magnitudes will dominate the decision function. Always standardize (or normalize) features before training an SVM.

In [ ]:
from sklearn.preprocessing import MinMaxScaler

X_wine, y_wine = load_wine(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X_wine, y_wine, test_size=0.3, random_state=42)

# Without scaling
svm_no_scale = SVC(kernel='rbf', gamma='scale', C=1)
svm_no_scale.fit(X_train, y_train)
acc_no_scale = svm_no_scale.score(X_test, y_test)

# With scaling
pipe_scaled = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(kernel='rbf', gamma='scale', C=1))
])
pipe_scaled.fit(X_train, y_train)
acc_scaled = pipe_scaled.score(X_test, y_test)

print(f'Accuracy without scaling:  {acc_no_scale:.3f}')
print(f'Accuracy with scaling:     {acc_scaled:.3f}')
print(f'Improvement:               {acc_scaled - acc_no_scale:+.3f}')

---
## 10. Decision Function and Probability Calibration

`SVC.decision_function()` returns signed distances to the hyperplane. For probability estimates, use `probability=True` (enables Platt scaling via cross-validation, which is computationally expensive).

In [ ]:
X_cancer, y_cancer = load_breast_cancer(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(X_cancer, y_cancer, test_size=0.3, random_state=42)

scaler = StandardScaler()
X_tr_scaled = scaler.fit_transform(X_tr)
X_te_scaled = scaler.transform(X_te)

svm_prob = SVC(kernel='rbf', gamma='scale', C=1, probability=True)
svm_prob.fit(X_tr_scaled, y_tr)

decisions = svm_prob.decision_function(X_te_scaled[:5])
probas = svm_prob.predict_proba(X_te_scaled[:5])

print('Decision function values (signed distance):')
for i, (dec, prob) in enumerate(zip(decisions, probas)):
    print(f'  Sample {i+1}: dist={dec:+.3f}  P(class 0)={prob[0]:.3f}  P(class 1)={prob[1]:.3f}  pred={svm_prob.classes_[np.argmax(prob)]}')

---
## 11. GridSearchCV for Hyperparameter Tuning

In [ ]:
X_digits, y_digits = load_digits(n_class=5, return_X_y=True)  # use 0-4 for speed
X_tr, X_te, y_tr, y_te = train_test_split(X_digits, y_digits, test_size=0.3, random_state=42)

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(kernel='rbf'))
])

param_grid = {
    'svm__C': [0.1, 1, 10, 100],
    'svm__gamma': [0.001, 0.01, 0.1, 1],
    'svm__kernel': ['rbf']
}

grid = GridSearchCV(pipe, param_grid, cv=5, scoring='accuracy', n_jobs=-1, verbose=0)
grid.fit(X_tr, y_tr)

print(f'Best params: {grid.best_params_}')
print(f'Best CV accuracy: {grid.best_score_:.4f}')
print(f'Test accuracy: {grid.score(X_te, y_te):.4f}')

---
## 12. Parameter Heatmap (C vs Gamma)

In [ ]:
results = grid.cv_results_
C_values = [0.1, 1, 10, 100]
gamma_values = [0.001, 0.01, 0.1, 1]
scores = results['mean_test_score'].reshape(len(C_values), len(gamma_values))

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(scores, interpolation='nearest', cmap='viridis', aspect='auto')
ax.set_xticks(np.arange(len(gamma_values)))
ax.set_yticks(np.arange(len(C_values)))
ax.set_xticklabels(gamma_values)
ax.set_yticklabels(C_values)
ax.set_xlabel('Gamma')
ax.set_ylabel('C')

for i in range(len(C_values)):
    for j in range(len(gamma_values)):
        ax.text(j, i, f'{scores[i, j]:.3f}', ha='center', va='center',
                color='white' if scores[i, j] < scores.max() * 0.7 else 'black', fontsize=10)

ax.set_title('Validation Accuracy: C vs Gamma (RBF Kernel)')
plt.colorbar(im, ax=ax, label='Accuracy')
plt.tight_layout()
plt.show()

---
## 13. Real-World Example: Wine Classification

In [ ]:
wine = load_wine()
X, y = wine.data, wine.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

best_svm = SVC(C=grid.best_params_['svm__C'],
               gamma=grid.best_params_['svm__gamma'],
               kernel='rbf', probability=True)
best_svm.fit(X_train, y_train)
y_pred = best_svm.predict(X_test)

print('Classification Report:')
print(classification_report(y_test, y_pred, target_names=wine.target_names))
print(f'Accuracy: {accuracy_score(y_test, y_pred):.4f}')
print(f'F1 (weighted): {f1_score(y_test, y_pred, average="weighted"):.4f}')

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5.5))
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=wine.target_names)
disp.plot(ax=ax, cmap='Blues', colorbar=True, values_format='d')
ax.set_title('Confusion Matrix - Wine Dataset (SVM with RBF Kernel)')
plt.tight_layout()
plt.show()

---
## 14. Support Vectors Visualization

In [ ]:
# Project onto first 2 PCA components to visualize SVs
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_train)

svm_vis = SVC(kernel='rbf', C=1, gamma='scale')
svm_vis.fit(X_pca, y_train)

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# Decision boundary
plot_decision_boundary(svm_vis, X_pca, y_train, axes[0],
                       'Decision Boundary (PCA-reduced Wine Data)')

# Support vectors highlighted
x_min, x_max = X_pca[:, 0].min() - 0.5, X_pca[:, 0].max() + 0.5
y_min, y_max = X_pca[:, 1].min() - 0.5, X_pca[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300),
                     np.linspace(y_min, y_max, 300))
Z = svm_vis.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)
axes[1].contourf(xx, yy, Z, alpha=0.3, cmap='coolwarm')
axes[1].scatter(X_pca[:, 0], X_pca[:, 1], c=y_train, cmap='coolwarm', edgecolors='k', s=40)
axes[1].scatter(svm_vis.support_vectors_[:, 0], svm_vis.support_vectors_[:, 1],
                s=180, facecolors='none', edgecolors='k', linewidth=2,
                label=f'Support Vectors ({len(svm_vis.support_vectors_)} pts)')
axes[1].set_xlim(xx.min(), xx.max())
axes[1].set_ylim(yy.min(), yy.max())
axes[1].set_title('Support Vectors Highlighted')
axes[1].set_xlabel('PC1')
axes[1].set_ylabel('PC2')
axes[1].legend(fontsize=9)

plt.suptitle(f'SVs make up {len(svm_vis.support_vectors_)/len(X_pca)*100:.1f}% of training data', fontsize=13)
plt.tight_layout()
plt.show()

---
## 15. Pros and Cons of SVM

**Advantages**
- Effective in high-dimensional spaces (e.g., text classification)
- Memory efficient — only uses support vectors (a subset of training points)
- Versatile via different kernel functions
- Robust against overfitting in high dimensions (max-margin principle)

**Disadvantages**
- Does not scale well to large datasets ($O(n^2)$ to $O(n^3)$ training time)
- Sensitive to feature scaling
- No direct probability estimates (requires Platt scaling)
- Performance highly dependent on kernel choice and hyperparameters
- Black-box model — decision function is not easily interpretable
- Hard to choose the right kernel

---
## 16. Comparison with Other Classifiers

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB

classifiers = {
    'Logistic Regression': LogisticRegression(max_iter=5000),
    'Decision Tree': DecisionTreeClassifier(max_depth=5),
    'Random Forest': RandomForestClassifier(n_estimators=100),
    'KNN (k=5)': KNeighborsClassifier(n_neighbors=5),
    'Naive Bayes': GaussianNB(),
    'SVM (RBF)': SVC(kernel='rbf', gamma='scale', C=1)
}

results_comp = {}
for name, clf in classifiers.items():
    pipe_clf = Pipeline([('scaler', StandardScaler()), ('clf', clf)])
    scores = cross_val_score(pipe_clf, X_train, y_train, cv=5, scoring='accuracy')
    pipe_clf.fit(X_train, y_train)
    test_acc = pipe_clf.score(X_test, y_test)
    results_comp[name] = {'CV Mean': scores.mean(), 'CV Std': scores.std(), 'Test Acc': test_acc}

print(f'{"Classifier":<22} {"CV Mean":<10} {"CV Std":<10} {"Test Acc":<10}')
print('-' * 52)
for name, res in results_comp.items():
    print(f'{name:<22} {res["CV Mean"]:.4f}    {res["CV Std"]:.4f}    {res["Test Acc"]:.4f}')

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
names = list(results_comp.keys())
cv_means = [results_comp[n]['CV Mean'] for n in names]
cv_stds = [results_comp[n]['CV Std'] for n in names]
test_accs = [results_comp[n]['Test Acc'] for n in names]

x = np.arange(len(names))
width = 0.35
bars1 = ax.bar(x - width/2, cv_means, width, yerr=cv_stds, capsize=4,
               label='CV Accuracy', color='steelblue', alpha=0.85)
bars2 = ax.bar(x + width/2, test_accs, width, label='Test Accuracy',
               color='coral', alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(names, rotation=25, ha='right', fontsize=10)
ax.set_ylabel('Accuracy')
ax.set_title('Classifier Comparison on Wine Dataset')
ax.legend(fontsize=10)
ax.set_ylim(0.5, 1.05)
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.008,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.008,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)
plt.tight_layout()
plt.show()

---
## Practice Exercises

Test your understanding of Support Vector Machines with these exercises.

### Exercise 1: Effect of C on Misclassification
Train a linear SVM on `make_blobs(n_samples=100, centers=2, cluster_std=3, random_state=1)`. Try `C = [0.001, 0.01, 0.1, 1, 10, 100]`. Plot a graph of `C` vs training accuracy and `C` vs number of support vectors. What pattern do you observe?

### Exercise 2: RBF Gamma on a Real Dataset
Using the breast cancer dataset, train an RBF SVM with `C=10`. Vary `gamma` across `[0.0001, 0.001, 0.01, 0.1, 1, 10]`. Plot training accuracy vs test accuracy as a function of gamma. At what gamma value does overfitting begin?

### Exercise 3: Kernel Comparison with GridSearch
Run `GridSearchCV` on the iris dataset comparing `linear`, `poly` (degree 2–4), and `rbf` kernels. Use `C = [0.1, 1, 10, 100]` and `gamma = ['scale', 'auto', 0.01, 0.1]`. Which kernel + parameters achieve the best accuracy?

### Exercise 4: SVR on a Noisy Sine Wave
Generate `y = sin(x) + N(0, 0.15)` for `x in [0, 4π]`. Train SVR with RBF kernel. Vary `epsilon` in `[0.01, 0.05, 0.1, 0.5, 1]`. Plot the predictions. How does epsilon affect the number of support vectors and the smoothness of the fit?

### Exercise 5: PCA + SVM for Image Classification
Using the full digits dataset (10 classes), reduce dimensionality with PCA to 20 components. Train an RBF SVM and tune `C` and `gamma` with `GridSearchCV`. Compare the accuracy and training time with an SVM trained on the original 64 features.